In [18]:
!pip install pandas

In [19]:
!pip install lxml

In [20]:
import pandas as pd

tables = pd.read_html("https://en.wikipedia.org/wiki/List_of_S&P_500_companies")


In [21]:
sp500_table = tables[0]

In [22]:
df = sp500_table[['Symbol', 'Security', 'Date added']]

In [23]:
df.head()

,Symbol,Security,Date added
0,MMM,3M,1957-03-04
1,AOS,A. O. Smith,2017-07-26
2,ABT,Abbott Laboratories,1957-03-04
3,ABBV,AbbVie,2012-12-31
4,ACN,Accenture,2011-07-06


In [24]:
from datetime import datetime
df['Date added'] = pd.to_datetime(df['Date added'], errors='coerce')
df['Year added'] = df['Date added'].dt.year


/var/folders/f1/xm7l5z8n71x3s0xdf5072rfr0000gn/T/ipykernel_95601/2067470023.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Date added'] = pd.to_datetime(df['Date added'], errors='coerce')
/var/folders/f1/xm7l5z8n71x3s0xdf5072rfr0000gn/T/ipykernel_95601/2067470023.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Year added'] = df['Date added'].dt.year


In [25]:
# rename the columns for clarity
df.rename(columns={
    'Symbol': 'Ticker',
    'Security': 'Company Name'
}, inplace=True)


/var/folders/f1/xm7l5z8n71x3s0xdf5072rfr0000gn/T/ipykernel_95601/3529570695.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.rename(columns={


In [26]:
df.head()

,Ticker,Company Name,Date added,Year added
0,MMM,3M,1957-03-04,1957
1,AOS,A. O. Smith,2017-07-26,2017
2,ABT,Abbott Laboratories,1957-03-04,1957
3,ABBV,AbbVie,2012-12-31,2012
4,ACN,Accenture,2011-07-06,2011


<h2> Question 1 <h2>

In [17]:
year_counts = df.groupby('Year added').size().reset_index(name='Count').sort_values(by='Count',ascending=False)
year_counts

,Year added,Count
0,1957,53
48,2017,23
47,2016,23
50,2019,22
39,2008,17
55,2024,16
53,2022,16
54,2023,15
52,2021,15
49,2018,14


<h3> The answers for q1 are 2017 and 2016. Both years have 23 companies joined SP 500 </h3>

<h2> Question 2 </h2>

In [27]:
!pip install yfinance


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 30.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 34.7 MB/s eta 0:00:00
  Created wheel for peewee: filename=peewee-3.18.1-cp311-cp311-macosx_11_0_arm64.whl size=264575 sha256=266d0f4121302cbfe5c2392bea9cc2fd2e63615420c21a56a491a21aa9ff7c40
  Stored in directory: /Users/cocochen/Library/Caches/pip/wheels/25/cb/79/a133a0d1d75f318a96614ed7fb97bdf2f35a7b6c4d4e426e3f
Successfully built peewee


In [28]:
import yfinance as yf

# Define tickers and index names
indices = {
    "^GSPC": "United States - S&P 500",
    "000001.SS": "China - Shanghai Composite",
    "^HSI": "Hong Kong - Hang Seng",
    "^AXJO": "Australia - S&P/ASX 200",
    "^NSEI": "India - Nifty 50",
    "^GSPTSE": "Canada - S&P/TSX Composite",
    "^GDAXI": "Germany - DAX",
    "^FTSE": "UK - FTSE 100",
    "^N225": "Japan - Nikkei 225",
    "^MXX": "Mexico - IPC",
    "^BVSP": "Brazil - Ibovespa"
}

start_date = "2025-01-01"
end_date = "2025-05-01"


In [30]:

# Download data
data = yf.download(list(indices.keys()), start=start_date, end=end_date)["Close"]


[*********************100%***********************]  11 of 11 completed


In [34]:

# Calculate YTD returns
# returns = {}
# for ticker in data.columns:
#     first_price = data[ticker].iloc[0]
#     last_price = data[ticker].iloc[-1]
#     ytd_return = ((last_price - first_price) / first_price) * 100
#     returns[indices[ticker]] = ytd_return

returns = {}
for ticker in data.columns:
    series = data[ticker].dropna()
    if not series.empty:
        first_price = series.iloc[0]
        last_price = series.iloc[-1]
        ytd_return = ((last_price - first_price) / first_price) * 100
        returns[indices[ticker]] = ytd_return
    else:
        returns[indices[ticker]] = None



In [35]:
# Convert to DataFrame
df = pd.DataFrame(returns.items(), columns=["Index", "YTD Return (%)"])
df.sort_values("YTD Return (%)", ascending=False, inplace=True)


In [36]:

# Count how many outperformed the S&P 500
sp500_return = returns["United States - S&P 500"]
better_count = sum(df["YTD Return (%)"] > sp500_return)

print(df)
print(f"\nIndexes with better YTD return than S&P 500: {better_count}")

                         Index  YTD Return (%)
8                 Mexico - IPC       13.049444
7        Hong Kong - Hang Seng       12.720018
2            Brazil - Ibovespa       12.438710
4                Germany - DAX       12.346378
3                UK - FTSE 100        2.842590
10            India - Nifty 50        2.490424
0   China - Shanghai Composite        0.504817
6   Canada - S&P/TSX Composite       -0.226126
1      Australia - S&P/ASX 200       -0.914500
5      United States - S&P 500       -5.103301
9           Japan - Nikkei 225       -8.297931

Indexes with better YTD return than S&P 500: 9


<h2> Question 3 </h2>

In [49]:
# sp500 = yf.download("^GSPC", start="1950-01-01")["Close"].dropna()
sp500 = yf.download("^GSPC", start="1950-01-01")[["Close"]]

sp500.index = pd.to_datetime(sp500.index)


[*********************100%***********************]  1 of 1 completed


In [50]:
all_time_highs = sp500.cummax()

# Step 3: Identify all-time high dates
ath_dates = sp500[sp500 == all_time_highs].index

In [51]:
print(type(sp500))  


<class 'pandas.core.frame.DataFrame'>


In [52]:
corrections = []
for i in range(len(ath_dates) - 1):
    start_date = ath_dates[i]
    end_date = ath_dates[i + 1]
    interim = sp500[start_date:end_date]
    if interim.empty:
        continue
    trough_price = interim.min()
    trough_date = interim.idxmin()
    # peak_price = sp500[start_date]
    nearest_date = sp500.index.asof(start_date)
    peak_price = float(sp500.loc[nearest_date, "Close"])

    drawdown = (peak_price - trough_price) / peak_price * 100
    if drawdown >= 5:
        duration = (end_date - start_date).days
        corrections.append({
            "Start": start_date,
            "End": end_date,
            "Bottom": trough_date,
            "Drawdown (%)": drawdown,
            "Duration (days)": duration
        })


/var/folders/f1/xm7l5z8n71x3s0xdf5072rfr0000gn/T/ipykernel_95601/564648643.py:12: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  peak_price = float(sp500.loc[nearest_date, "Close"])


ValueError: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

[*********************100%***********************]  1 of 1 completed


ValueError: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

In [ ]:
# Step 4: Create DataFrame and calculate percentiles
df = pd.DataFrame(corrections)
print(df)
print("\nCorrection duration percentiles (days):")
print(df["Duration (days)"].quantile([0.25, 0.5, 0.75]))

In [ ]:

# Step 1: download just the close column as Series
close_series = yf.download("^GSPC", start="1950-01-01")["Close"]
close_series.index = pd.to_datetime(close_series.index)

# Step 2: get ATHs
all_time_highs = close_series.cummax()
ath_dates = close_series[close_series == all_time_highs].index

# Step 3: loop through corrections
corrections = []

for i in range(len(ath_dates) - 1):
    start_date = ath_dates[i]
    end_date = ath_dates[i + 1]
    interim = close_series[start_date:end_date]

    if interim.empty:
        continue

    trough_price = interim.min()
    trough_date = interim.idxmin()

    nearest_date = close_series.index.asof(start_date)
    if pd.isna(nearest_date):
        continue

    peak_price = close_series.loc[nearest_date]  # guaranteed to be float (Series)

    drawdown = (peak_price - trough_price) / peak_price * 100

    if drawdown >= 5:
        duration = (end_date - start_date).days
        corrections.append({
            "Start": start_date,
            "End": end_date,
            "Bottom": trough_date,
            "Drawdown (%)": round(drawdown, 2),
            "Duration (days)": duration
        })

# Step 4: results
df = pd.DataFrame(corrections)
print(df)

print("\n📊 Correction Duration Percentiles (in days):")
print(df["Duration (days)"].quantile([0.25, 0.5, 0.75]))
